# NB_MDP_00 · Orquestador

Lo llama el pipeline `PL_MDP_MEDALLION` después de las copias. Recibe en `config_json` la salida de `meta.usp_GET_RUN_CONFIG` y ejecuta los notebooks hijos con `notebookutils.notebook.runMultiple`, en la **misma sesión de Spark** (no levanta un clúster por entidad).

```text
Fase 1 (un DAG):  bronze_<entidad> ──► silver_<entidad>     (todas las entidades en paralelo)
Fase 2 (otro DAG): gold_<objeto>, respetando DEPENDS_ON_GOLD (solo si la fase 1 no tuvo errores)
```

Devuelve con `notebook.exit` un JSON con el resultado de cada paso. El pipeline se lo pasa a `log.usp_END_RUN`, que registra los pasos y avanza los watermarks.

Para reprocesar silver y gold de una corrida anterior sin volver a copiar: `layers = "silver,gold"` y `reprocesar_run_id = "<run_id viejo>"`.

In [ ]:
# Parámetros
run_id = "manual"
config_json = "{}"
layers = "bronze,silver,gold"
reprocesar_run_id = ""          # silver lee de bronze las filas de esta corrida en vez de la actual
concurrencia = 4                # notebooks en paralelo dentro de la sesión
timeout_celda_seg = 3600        # el default de runMultiple es 90 segundos por celda: demasiado poco
gold_si_hay_errores = False     # True = corre gold aunque alguna entidad haya fallado

In [ ]:
import json, re

NB_BRONZE, NB_SILVER, NB_GOLD = "NB_MDP_10_bronze", "NB_MDP_20_silver", "NB_MDP_30_gold"

cfg = json.loads(config_json)
capas = {c.strip().lower() for c in layers.split(",") if c.strip()}
entidades = cfg.get("entities") or []
golds = cfg.get("gold_objects") or []
pasos = {}      # nombre de actividad -> resultado


def act(prefijo, codigo):
    return f"{prefijo}_{re.sub(r'[^0-9A-Za-z_]', '_', codigo)}"


def paso_vacio(layer, codigo, status, error=None):
    return {"layer": layer, "object_code": codigo, "status": status, "error": error}


def correr(actividades, meta_por_actividad):
    """Ejecuta un DAG y devuelve {actividad: resultado}. Los hijos nunca lanzan excepción
    (devuelven status FAILED), así que una excepción acá es de infraestructura: timeout,
    notebook inexistente o sesión caída."""
    if not actividades:
        return {}
    dag = {"activities": actividades, "timeoutInSeconds": 43200, "concurrency": int(concurrencia)}
    notebookutils.notebook.validateDAG(dag)
    salida, error_general = {}, None
    try:
        crudo = notebookutils.notebook.runMultiple(dag, {"displayDAGViaGraphviz": False})
    except Exception as ex:
        error_general = f"{type(ex).__name__}: {ex}"[:3000]
        crudo = getattr(ex, "result", None) or {}
    for a in actividades:
        layer, codigo = meta_por_actividad[a["name"]]
        r = crudo.get(a["name"]) if isinstance(crudo, dict) else None
        try:
            if r and r.get("exception") is None and r.get("exitVal"):
                salida[a["name"]] = json.loads(r["exitVal"])
                continue
            motivo = str(r.get("exception")) if r and r.get("exception") is not None else error_general
        except Exception as ex:
            motivo = f"Salida ilegible del notebook: {ex}"
        salida[a["name"]] = paso_vacio(layer, codigo, "FAILED", motivo or "El notebook no devolvió resultado")
    return salida

In [ ]:
# La tabla de rechazos la comparten todas las entidades: se crea antes para que dos
# notebooks en paralelo no intenten crearla a la vez
if "silver" in capas:
    for esquema in {".".join(e["silver_table"].split(".")[:-1]) for e in entidades}:
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {esquema}.dq_rechazos
                      (entity_code STRING, rule_name STRING, run_id STRING, row_json STRING, rejected_at TIMESTAMP)
                      USING DELTA""")

# Fase 1: bronze -> silver por entidad
actividades, meta_act = [], {}
for e in entidades:
    codigo, ej = e["entity_code"], json.dumps(e, ensure_ascii=False)
    if "bronze" in capas:
        nombre = act("bronze", codigo)
        actividades.append({"name": nombre, "path": NB_BRONZE, "timeoutPerCellInSeconds": int(timeout_celda_seg),
                            "args": {"run_id": run_id, "entity_json": ej}})
        meta_act[nombre] = ("bronze", codigo)
    if "silver" in capas:
        nombre = act("silver", codigo)
        actividades.append({"name": nombre, "path": NB_SILVER, "timeoutPerCellInSeconds": int(timeout_celda_seg),
                            "args": {"run_id": run_id, "entity_json": ej, "bronze_run_id": reprocesar_run_id},
                            "dependencies": [act("bronze", codigo)] if "bronze" in capas else []})
        meta_act[nombre] = ("silver", codigo)

pasos.update(correr(actividades, meta_act))

# Si bronze falló, el silver de esa entidad no encontró datos: se registra como bloqueado, no como "sin datos"
for e in entidades:
    b, s = pasos.get(act("bronze", e["entity_code"])), pasos.get(act("silver", e["entity_code"]))
    if b and s and b["status"] in ("FAILED", "SKIPPED") and s["status"] == "SKIPPED_NO_DATA":
        s["status"] = "SKIPPED_UPSTREAM_FAILED"

errores_fase1 = [k for k, v in pasos.items() if v["status"] == "FAILED"]
print(f"Fase 1: {len(actividades)} actividades, {len(errores_fase1)} con error {errores_fase1}")

In [ ]:
# Fase 2: gold
if "gold" in capas and golds:
    if errores_fase1 and not gold_si_hay_errores:
        for g in golds:
            pasos[act("gold", g["gold_code"])] = paso_vacio(
                "gold", g["gold_code"], "SKIPPED_UPSTREAM_FAILED",
                "Gold no se ejecutó porque hubo errores en bronze o silver (gold_si_hay_errores = False)")
    else:
        activos = {g["gold_code"] for g in golds}
        actividades, meta_act = [], {}
        for g in golds:
            deps = [d.strip() for d in (g.get("depends_on_gold") or "").split(",") if d.strip() in activos]
            nombre = act("gold", g["gold_code"])
            actividades.append({"name": nombre, "path": NB_GOLD, "timeoutPerCellInSeconds": int(timeout_celda_seg),
                                "args": {"run_id": run_id, "gold_json": json.dumps(g, ensure_ascii=False)},
                                "dependencies": [act("gold", d) for d in deps]})
            meta_act[nombre] = ("gold", g["gold_code"])
        pasos.update(correr(actividades, meta_act))

resumen = {
    "run_id": run_id,
    "hay_errores": any(v["status"] == "FAILED" for v in pasos.values()),
    "pasos": list(pasos.values()),
}
display(spark.createDataFrame([
    (p["layer"], p["object_code"], p["status"], str(p.get("rows_written")), (p.get("error") or "")[:200])
    for p in resumen["pasos"]], "layer string, object_code string, status string, rows_written string, error string"))

In [ ]:
notebookutils.notebook.exit(json.dumps(resumen, ensure_ascii=False, default=str))